# Week 2 Day 4 — Single Patient Prediction Pipeline

## ICU Early Warning Prediction System

In Week 2 Day 3, the final Gradient Boosting + SMOTE model was saved using `joblib`.

Today, the objective is to build a reusable single-patient prediction pipeline.

This notebook simulates how the ICU Early Warning Prediction System would work in deployment:

1. load the saved model
2. load the required feature names
3. select one ICU patient
4. predict sepsis probability
5. convert probability into a risk category
6. generate a clinical alert message

This is an important step toward dashboard and API deployment.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Models path:", MODELS_DIR)
print("Results path:", RESULTS_DIR)

Project root: c:\Users\User\OneDrive\Desktop\icu-early-warning-system
Data path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\data\processed
Models path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\models
Results path: c:\Users\User\OneDrive\Desktop\icu-early-warning-system\results


In [3]:
model_path = MODELS_DIR / "gradient_boosting_smote_model.joblib"
feature_names_path = MODELS_DIR / "model_feature_names.joblib"
metadata_path = MODELS_DIR / "model_metadata.joblib"

model = joblib.load(model_path)
feature_names = joblib.load(feature_names_path)
metadata = joblib.load(metadata_path)

print("Saved model loaded successfully.")
print("Model name:", metadata["model_name"])
print("Number of features:", len(feature_names))

Saved model loaded successfully.
Model name: Gradient Boosting + SMOTE
Number of features: 16


In [4]:
data_file = DATA_PATH / "day2_patient_level_features.csv"

df = pd.read_csv(data_file)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100, 18)


,Patient_ID,HR_mean,HR_max,HR_min,O2Sat_mean,O2Sat_min,Temp_mean,Temp_max,SBP_mean,SBP_min,MAP_mean,MAP_min,Resp_mean,Resp_max,Age_first,Gender_first,ICULOS_max,SepsisLabel_max
0,p000001,101.907407,117.0,76.0,91.453704,85.0,36.735185,37.44,127.870370,78.0,88.321111,44.00,24.555556,32.0,83.14,0,54,0
1,p000002,62.173913,94.0,54.0,97.043478,94.0,36.206087,37.00,129.043478,114.0,67.239130,50.50,14.630435,27.0,75.91,0,23,0
2,p000003,79.968750,93.0,68.0,95.375000,91.0,37.465000,38.61,139.760417,121.0,81.149167,62.67,25.302083,40.0,45.82,0,48,0
3,p000004,102.172414,113.0,89.0,98.189655,95.5,36.463103,37.00,113.017241,90.0,67.063103,34.00,18.758621,26.0,65.71,0,29,0
4,p000005,76.604167,88.0,61.0,97.677083,96.0,37.072292,37.33,135.072917,114.0,90.364583,73.00,15.447917,21.0,28.09,1,49,0


In [5]:
target_col = "SepsisLabel_max"

X = df[feature_names]
y = df[target_col]
patient_ids = df["Patient_ID"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (100, 16)
Target shape: (100,)


In [6]:
def categorize_risk(probability):
    risk_score = probability * 100

    if risk_score < 25:
        return "Low Risk"
    elif risk_score < 50:
        return "Moderate Risk"
    elif risk_score < 75:
        return "High Risk"
    else:
        return "Critical Risk"

In [7]:
def generate_clinical_alert(risk_category):
    if risk_category == "Low Risk":
        return "Routine monitoring recommended."
    elif risk_category == "Moderate Risk":
        return "Increase monitoring frequency and reassess patient condition."
    elif risk_category == "High Risk":
        return "Clinical review recommended due to elevated sepsis risk."
    else:
        return "Urgent clinical review recommended due to critical sepsis risk."

In [9]:
def predict_single_patient(patient_row, patient_id=None, true_label=None):
    """
    Predict sepsis risk for a single ICU patient.
    """

    patient_row = patient_row[feature_names]

    probability = model.predict_proba(patient_row)[0, 1]
    predicted_label = model.predict(patient_row)[0]

    risk_score = probability * 100
    risk_category = categorize_risk(probability)
    clinical_alert = generate_clinical_alert(risk_category)

    result = {
        "Patient_ID": patient_id,
        "True_Sepsis_Label": true_label,
        "Predicted_Sepsis_Label": int(predicted_label),
        "Sepsis_Probability": probability,
        "Risk_Score_Percent": risk_score,
        "Risk_Category": risk_category,
        "Clinical_Alert": clinical_alert
    }

    return result

In [10]:
patient_index = 0

patient_row = X.iloc[[patient_index]]
patient_id = patient_ids.iloc[patient_index]
true_label = y.iloc[patient_index]

single_patient_result = predict_single_patient(
    patient_row=patient_row,
    patient_id=patient_id,
    true_label=true_label
)

single_patient_result

{'Patient_ID': 'p000001',
 'True_Sepsis_Label': np.int64(0),
 'Predicted_Sepsis_Label': 0,
 'Sepsis_Probability': np.float64(0.0012298665225789678),
 'Risk_Score_Percent': np.float64(0.12298665225789678),
 'Risk_Category': 'Low Risk',
 'Clinical_Alert': 'Routine monitoring recommended.'}

In [11]:
single_patient_df = pd.DataFrame([single_patient_result])

single_patient_df

,Patient_ID,True_Sepsis_Label,Predicted_Sepsis_Label,Sepsis_Probability,Risk_Score_Percent,Risk_Category,Clinical_Alert
0,p000001,0,0,0.00123,0.122987,Low Risk,Routine monitoring recommended.


In [12]:
single_patient_df.to_csv(
    RESULTS_DIR / "day11_single_patient_prediction.csv",
    index=False
)

print("Single patient prediction saved successfully.")

Single patient prediction saved successfully.


In [13]:
example_results = []

for patient_index in range(10):
    patient_row = X.iloc[[patient_index]]
    patient_id = patient_ids.iloc[patient_index]
    true_label = y.iloc[patient_index]

    result = predict_single_patient(
        patient_row=patient_row,
        patient_id=patient_id,
        true_label=true_label
    )

    example_results.append(result)

example_predictions_df = pd.DataFrame(example_results)

example_predictions_df

,Patient_ID,True_Sepsis_Label,Predicted_Sepsis_Label,Sepsis_Probability,Risk_Score_Percent,Risk_Category,Clinical_Alert
0,p000001,0,0,0.001230,0.122987,Low Risk,Routine monitoring recommended.
1,p000002,0,0,0.006569,0.656914,Low Risk,Routine monitoring recommended.
2,p000003,0,0,0.036493,3.649337,Low Risk,Routine monitoring recommended.
3,p000004,0,0,0.002997,0.299668,Low Risk,Routine monitoring recommended.
4,p000005,0,0,0.000846,0.084589,Low Risk,Routine monitoring recommended.
5,p000006,0,0,0.001154,0.115413,Low Risk,Routine monitoring recommended.
6,p000007,0,0,0.000408,0.040844,Low Risk,Routine monitoring recommended.
7,p000008,0,0,0.000157,0.015665,Low Risk,Routine monitoring recommended.
8,p000009,1,1,0.999256,99.925559,Critical Risk,Urgent clinical review recommended due to crit...
9,p000010,0,0,0.001761,0.176097,Low Risk,Routine monitoring recommended.


In [14]:
example_predictions_df.to_csv(
    RESULTS_DIR / "day11_example_patient_predictions.csv",
    index=False
)

print("Example patient predictions saved successfully.")

Example patient predictions saved successfully.


In [15]:
example_predictions_df[
    example_predictions_df["Risk_Category"].isin(["High Risk", "Critical Risk"])
]

,Patient_ID,True_Sepsis_Label,Predicted_Sepsis_Label,Sepsis_Probability,Risk_Score_Percent,Risk_Category,Clinical_Alert
8,p000009,1,1,0.999256,99.925559,Critical Risk,Urgent clinical review recommended due to crit...


## Clinical Interpretation

This notebook created a reusable single-patient prediction pipeline.

The model now produces more than a binary classification.

For each patient, the system returns:

- predicted sepsis label
- sepsis probability
- risk score percentage
- risk category
- clinical alert message

This makes the model easier to use in a future dashboard or API.

Instead of only saying whether sepsis is predicted, the system gives a clinically understandable warning level.

In [16]:
print("Week 2 Day 4 completed successfully.")
print("Generated outputs:")
print("- day11_single_patient_prediction.csv")
print("- day11_example_patient_predictions.csv")

Week 2 Day 4 completed successfully.
Generated outputs:
- day11_single_patient_prediction.csv
- day11_example_patient_predictions.csv
